<a href="https://colab.research.google.com/github/Steve-Falkovsky/Hypencoder-Entity-Linking/blob/main/notebooks/Hypencoder_Vs_fine_tuned_Hypencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Environment lock (run once per fresh runtime) ---
# because libraries like torch can change and make your code non-runnable :)

TORCH = "2.9.0"
TRANSFORMERS = "5.0.0"
ACCELERATE = "1.12.0"
SAFETENSORS = "0.7.0"

%pip install -q --upgrade \
  torch=={TORCH} \
  transformers=={TRANSFORMERS} \
  accelerate=={ACCELERATE} \
  safetensors=={SAFETENSORS}

import os, sys
print("Installed. Restarting runtime is required.")

Installed. Restarting runtime is required.


In [2]:
import os

REPO_NAME = "Hypencoder-Entity-Linking"
GIT_URL = f"https://github.com/Steve-Falkovsky/{REPO_NAME}.git"
BRANCH_NAME = "main"

!git clone -b {BRANCH_NAME} --single-branch {GIT_URL}

# Move into the downloaded repo (The Root)
os.chdir(REPO_NAME)

%pip install -q -e "./hypencoder-paper"

os.chdir("hypencoder-paper")

print(f"📍 Working Directory is now: {os.getcwd()}")
print("✅ Environment Ready!")

Cloning into 'Hypencoder-Entity-Linking'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 445 (delta 42), reused 40 (delta 24), pack-reused 367 (from 1)
Receiving objects: 100% (445/445), 898.01 KiB | 6.41 MiB/s, done.
Resolving deltas: 100% (233/233), done.
  Preparing metadata (setup.py) ... done
📍 Working Directory is now: /content/Hypencoder-Entity-Linking/hypencoder-paper
✅ Environment Ready!


In [3]:
from datasets import load_dataset

# there are all "positive" pairs
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5424 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5445 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5516 [00:00<?, ? examples/s]

In [4]:
train_pairs = dataset['train']
print(train_pairs)

mention_names = train_pairs['mention']
mention_texts = train_pairs['mention_text']
entity_names = train_pairs['entity']
definitions = train_pairs['definition']

print(mention_names[:3])
print(entity_names[:3])

Dataset({
    features: ['mention', 'mention_text', 'entity', 'aliases', 'definition', 'id'],
    num_rows: 5424
})
['Naloxone', 'clonidine', 'hypertensive']
['Naloxone', 'Clonidine', 'Hypertension']


### Load the model

In [5]:
# Core Hypencoder model for outputing dense vector representations
from hypencoder_cb.modeling.hypencoder import Hypencoder, HypencoderDualEncoder, TextEncoder
from transformers import AutoTokenizer

model_name = "Stevenf232/SapBERT_freeze_hypencoder_hardneg"

dual_encoder = HypencoderDualEncoder.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


query_encoder: Hypencoder = dual_encoder.query_encoder
passage_encoder: TextEncoder = dual_encoder.passage_encoder

config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/483M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/483M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/438 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

### Move the model to the GPU

In [6]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'

# Move the model to the GPU
passage_encoder.to(device)
query_encoder.to(device)

Using device: cuda


Hypencoder(
  (transformer): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

### Load datasets and tokenise

In [14]:
# convert from type "datasets" to python list
queries = list(zip(mention_names, mention_texts))

passages = list(zip(entity_names, definitions))


# the output of the tokenizer contains 3 fields:
# input_ids, token_type_ids, and attention_mask
# all contain a tensor in the shape (number of queries, max number of tokens)

query_inputs = tokenizer(queries, return_tensors="pt", padding=True, truncation=True, max_length = 128)
passage_inputs = tokenizer(passages, return_tensors="pt", padding=True, truncation=True, max_length = 128)


In [15]:
print(f"query_inputs:\n{query_inputs}")
print("\n\n\n")
print(f"passage_inputs:\n{passage_inputs}")

query_inputs:
{'input_ids': tensor([[    2, 21926,     3,  ...,     0,     0,     0],
        [    2, 26596,     3,  ...,     0,     0,     0],
        [    2, 11706,     3,  ...,  1930, 21926,     3],
        ...,
        [    2, 19428,  1030,  ..., 17316,  1942,     3],
        [    2,  3692,     3,  ...,  1942,  1920,     3],
        [    2, 12948,  1019,  ..., 17316,  1942,     3]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])}




passage_inputs:
{'input_ids': tensor([[    2, 21926,     3,  ...,     0,     0,     0],
        [    2, 26596,     3,  ...,     0, 

# Passage Encodings


In [16]:
from tqdm import tqdm
import torch
from torch.amp import autocast

def batch_encode_passages(encoder ,passages):
  batch_size=256
  entity_name_features = []

  num_passages = passages["input_ids"].shape[0]

  with torch.no_grad(): # Disable gradient calculation (saves tons of memory)
    for i in tqdm(range(0, num_passages, batch_size), desc="Extracting features"):

        # extract entity features
        # Autocast does the math in fp16 where possible (default is fp32)
        # this will save memory and increase speed. The loss in precision shouldn't matter much (can check on a small sample if we want)
        with autocast("cuda"):
          features = encoder(
              input_ids=passages["input_ids"][i:i + batch_size].to(device),
              attention_mask=passages["attention_mask"][i:i + batch_size].to(device)
            ).representation

          entity_name_features.append(features.detach().cpu()) # Detach and move to CPU to save VRAM/RAM


  features_tensor = torch.cat(entity_name_features, dim=0)

  return features_tensor

In [17]:
passage_embeddings = batch_encode_passages(passage_encoder, passage_inputs)

Extracting features: 100%|██████████| 22/22 [00:07<00:00,  2.95it/s]


##Now, we create the q-nets.

For each q-net, we feed through it all the passages the calculate the similarity.

But the q-nets are created in batches, and every batch is represented as a single object `NoTorchSequential`. (Check out the `RepeatedDenseBlockConverter` class in q_net.py for more info)

This object expects an input in the shape (N, M, H):

* N = number of queries (mentions)

* M = number of passages (entities)

* H = Hidden dimension (e.g., 768 for BERT)



---



The passage embeddings have the shape (M, H) so we must create an additional dimension of size N.

This will be done like so:
`passages_batch = passages.unsqueeze(0).expand(num_queries, -1, -1)`

* `.unsqueeze()` adds a new dimension (in our case at location 0)

* `.expand()` "expands" that new dimension to be size "num_queries"

* `.expand()` creates a view, so it costs almost 0 memory! (compared to .repeat() which changes the tensor)

# Q-nets take **a lot** of memory.

Instead of creating all of them and then doing the similarity calculation, we will create batches and calculate similarities for just those q-nets, then discard those q-nets and move on to the next batch.

In [18]:
def batch_encode_queries(encoder, queries, passage_embeddings):
  batch_size = 8
  similarity_scores = []

  num_queries = queries["input_ids"].shape[0]

  with torch.no_grad():
    for i in tqdm(range(0, num_queries, batch_size), desc="Creating q-nets and calculating similarity scores"):

        # create q-nets
        with autocast("cuda"):
          q_nets = encoder(
              input_ids=queries["input_ids"][i:i + batch_size].to(device),
              attention_mask=queries["attention_mask"][i:i + batch_size].to(device)
            ).representation


        passages_gpu = passage_embeddings.to(device)

        # Note: we use q_nets.num_queries (our repo's noTorch equivalent of q_nets.shape[0]) instead of batch_size
        # because the total number might not be divisible by batch_size so the last batch might be smaller than the actual batch size
        passages_batch = passages_gpu.unsqueeze(0).expand(q_nets.num_queries, -1, -1)

        # calculate similarity
        batch_scores = q_nets(passages_batch)
        similarity_scores.append(batch_scores.detach().cpu())


  scores_tensor = torch.cat(similarity_scores, dim=0)
  return scores_tensor


In [19]:
similarity_scores = batch_encode_queries(query_encoder, query_inputs, passage_embeddings)

Creating q-nets and calculating similarity scores: 100%|██████████| 678/678 [00:43<00:00, 15.65it/s]


In [ ]:
# Case 1 - comparing a query to its respective passage

# In the simple case where each q_net only takes one passage, we can just
# reshape the passage_embeddings to (N, 1, H).
# passage_embeddings_single = passage_embeddings.unsqueeze(1)
# print(f"passage_embeddings shape: {passage_embeddings_single.shape}")
# giving the nueral network the input of passage_embeddings
# the output provides the relevance score of query 1 against passage 1, query 2 against passage 2, etc...
# scores = q_nets(passage_embeddings_single)
# print(f"scores: {scores}")

In [ ]:
# Case 2 - comparing a query to all passages

# The case where each q_net takes multiple passages
# meaning multiple passages are now associated with each of the queries

# this operation creates a 3D tensor which takes too much memory
# passage_embeddings_multi = passage_embeddings.repeat(N, 1).reshape(N, M, H)
# print(f"passage_embeddings shape: {passage_embeddings_multi.shape}")


# unbatched similarity scores for q-nets
# similarity_scores = q_nets(passage_embeddings_multi)
# print(f"similarity_scores shape: {similarity_scores.shape}")
#print(f"similarity_scores: {similarity_scores}")


In [20]:
similarity_scores

tensor([[[ 1.3846e+01],
         [ 1.0839e+01],
         [ 1.9955e+00],
         ...,
         [ 7.4531e-01],
         [ 2.8510e+00],
         [ 1.8613e-03]],

        [[ 1.1817e+01],
         [ 1.1627e+01],
         [ 2.5203e+00],
         ...,
         [ 4.0376e-01],
         [ 2.1408e+00],
         [-1.3987e-01]],

        [[ 9.8566e+00],
         [ 8.8256e+00],
         [ 4.4582e+00],
         ...,
         [ 2.4918e+00],
         [ 4.0907e+00],
         [ 2.4284e+00]],

        ...,

        [[ 3.9384e+00],
         [ 5.3116e+00],
         [ 4.7730e+00],
         ...,
         [ 7.5575e+00],
         [ 8.7849e+00],
         [ 6.6732e+00]],

        [[ 3.6382e+00],
         [ 4.0862e+00],
         [ 3.6953e+00],
         ...,
         [ 5.6022e+00],
         [ 8.6910e+00],
         [ 5.9941e+00]],

        [[ 3.8041e+00],
         [ 4.6951e+00],
         [ 4.4970e+00],
         ...,
         [ 5.4016e+00],
         [ 7.9381e+00],
         [ 7.7991e+00]]])

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def evaluate(train_pairs):
  correct_count = 0
  top_idxs = torch.argmax(similarity_scores,dim=1).flatten()

  for i in range(len(queries)):
      top_idx = top_idxs[i]
      # the conversion to int from here on out is because the original idx is of type numpy.int64
      top_match_id = train_pairs["id"][int(top_idx)]
      correct_id = train_pairs["id"][int(i)]

      if top_match_id == correct_id:
          correct_count += 1

      mention_name = train_pairs["mention"][int(i)]
      top_match = train_pairs["entity"][int(top_idx)]
      correct_name = train_pairs["entity"][int(i)]
      print(f"mention_name: {mention_name}\ncorrect entity name: {correct_name}\ntop_match: {top_match}\n")


  print(f"total comparisons: {len(queries)}")
  print(f"correct comparisons: {correct_count}")
  print(f"accuracy: {correct_count / len(queries)}")

In [22]:
evaluate(train_pairs)

Streaming output truncated to the last 5000 lines.

mention_name: Thyroid disorders
correct entity name: Thyroid Diseases
top_match: Atrial Flutter

mention_name: acute alcohol intoxication
correct entity name: Alcoholic Intoxication
top_match: Atrial Flutter

mention_name: atrial fibrillation
correct entity name: Atrial Fibrillation
top_match: Atrial Flutter

mention_name: fractures
correct entity name: Fractures, Bone
top_match: Atrial Flutter

mention_name: magnesium
correct entity name: Magnesium
top_match: Atrial Flutter

mention_name: potassium
correct entity name: Potassium
top_match: Atrial Flutter

mention_name: alcohol
correct entity name: Ethanol
top_match: Atrial Flutter

mention_name: creatine
correct entity name: Creatine
top_match: Atrial Flutter

mention_name: heparin
correct entity name: Heparin
top_match: Atrial Flutter

mention_name: diltiazem
correct entity name: Diltiazem
top_match: Atrial Flutter

mention_name: amiodarone
correct entity name: Amiodarone
top_match:

In [ ]:
# more evaluation methods
#from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
# print(f"{accuracy_score(train_labels, predicted_labels)=:.3f}")
# print(f"{recall_score(train_labels, predicted_labels)=:.3f}")
# print(f"{precision_score(train_labels, predicted_labels)=:.3f}")
# print(f"{f1_score(train_labels, predicted_labels)=:.3f}")